# Sparse and Transformer representations under one protocol

TF-IDF and frozen sentence embeddings are compared on the same stratified held-out split. Hyperparameters are selected with training folds only. Besides macro-F1, the experiment records encoding latency and a paired bootstrap interval over per-example correctness.

In [2]:
from time import perf_counter
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from nlp_portfolio.evaluation import paired_bootstrap_delta

categories = ["sci.space", "rec.sport.baseball"]
dataset = fetch_20newsgroups(subset="all", categories=categories,
                             remove=("headers", "footers", "quotes"))
texts = np.asarray(dataset.data, dtype=object)
y = np.asarray(dataset.target)
train_idx, test_idx = train_test_split(
    np.arange(len(y)), test_size=0.2, random_state=42, stratify=y
)
cv = StratifiedKFold(5, shuffle=True, random_state=42)
print("examples:", len(texts), "held out:", len(test_idx))

examples: 1981 held out: 397


In [3]:
tfidf = GridSearchCV(
    Pipeline([("features", TfidfVectorizer(min_df=2, sublinear_tf=True)),
              ("classifier", LogisticRegression(max_iter=1500))]),
    {"features__ngram_range": [(1, 1), (1, 2)], "classifier__C": [0.1, 1, 10]},
    scoring="f1_macro", cv=cv, n_jobs=-1,
)
start = perf_counter()
tfidf.fit(texts[train_idx].tolist(), y[train_idx])
tfidf_seconds = perf_counter() - start
tfidf_prediction = tfidf.predict(texts[test_idx].tolist())
print("TF-IDF macro-F1:", round(f1_score(y[test_idx], tfidf_prediction, average="macro"), 3))
print("fit plus encoding seconds:", round(tfidf_seconds, 2))

TF-IDF macro-F1: 0.934
fit plus encoding seconds: 8.45


## Frozen sentence encoder

The encoder is used as a fixed feature extractor. This isolates representation quality and compute cost from end-to-end fine-tuning.

In [5]:
from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
start = perf_counter()
embeddings = encoder.encode(texts.tolist(), batch_size=32, show_progress_bar=True,
                            normalize_embeddings=True)
encoding_seconds = perf_counter() - start
dense_head = GridSearchCV(
    LogisticRegression(max_iter=1500), {"C": [0.1, 1, 10]},
    scoring="f1_macro", cv=cv, n_jobs=-1,
)
dense_head.fit(embeddings[train_idx], y[train_idx])
dense_prediction = dense_head.predict(embeddings[test_idx])
print("frozen encoder macro-F1:", round(f1_score(y[test_idx], dense_prediction, average="macro"), 3))
print("one-time encoding seconds:", round(encoding_seconds, 2))

frozen encoder macro-F1: 0.962
one-time encoding seconds: 58.92


Batches: 100%|##########| 62/62 [00:58<00:00,  1.05it/s]


In [6]:
tfidf_correct = (tfidf_prediction == y[test_idx]).astype(float)
dense_correct = (dense_prediction == y[test_idx]).astype(float)
low, high = paired_bootstrap_delta(tfidf_correct, dense_correct, samples=5000, seed=42)
summary = pd.DataFrame([
    {"system": "TF-IDF", "macro_f1": f1_score(y[test_idx], tfidf_prediction, average="macro"),
     "correct": tfidf_correct.mean(), "encoding_seconds": tfidf_seconds},
    {"system": "Frozen MiniLM", "macro_f1": f1_score(y[test_idx], dense_prediction, average="macro"),
     "correct": dense_correct.mean(), "encoding_seconds": encoding_seconds},
])
display(summary.round(3))
print("95% interval for dense minus sparse accuracy:", (round(low, 3), round(high, 3)))

,system,macro_f1,correct,encoding_seconds
0,TF-IDF,0.934,0.935,8.451
1,Frozen MiniLM,0.962,0.962,58.920


95% interval for dense minus sparse accuracy: (0.005, 0.053)


The comparison is paired because both systems classify the same test documents. Latency values depend on hardware and caching, so they should be treated as run metadata. Fine-tuning a Transformer answers a different question and requires a separate compute budget and validation protocol.